Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
silver_table = f"{catalog}.{silver_schema}.complete_data"
gold_table = f"{catalog}.{gold_schema}.fact_flights"

Read silver Delta table into a DataFrame

In [0]:
from pyspark.sql import functions as F

silver_complete_df=(
    spark.read
    .format("delta")
    .table(silver_table)
    .filter(F.col("batch_id")==batch_id)
)

Create two airports lookup tables to transform departure and origin columns from str to int by joining them to the main fact table

In [0]:


silver_stations_df=(
    spark.read
    .format("delta")
    .table("airlines_lakehouse_2022.silver.stations")
)

origin_lookup_df=(
    silver_stations_df
    .select(
       F.col("airport_id").alias("origin_airport_id"),
       F.col("airport").alias("origin_airport")
    )
)

destination_lookup_df=(
    silver_stations_df
    .select(
       F.col("airport_id").alias("destination_airport_id"),
       F.col("airport").alias("destination_airport")
    )
)

In [0]:
silver_complete_df=(
    silver_complete_df
    .join(
        origin_lookup_df,
        silver_complete_df.origin_key==origin_lookup_df.origin_airport,
        "inner"
          )
    .join(
        destination_lookup_df,
        silver_complete_df.destination_key==destination_lookup_df.destination_airport,
        "inner"
          )
)


Select and rename columns for clarity and unification to create main fact table

In [0]:
gold_fact_flights_df=(
    silver_complete_df
    .select(
        "date_key",
        "departure_timestamp",
        "mkt_unique_carrier_key",
        "mkt_carrier_fl_num",
        "op_unique_carrier_key",
        "op_carrier_fl_num",
        "tail_number_key",
        F.col("origin_airport_id").alias("origin_airport_key"),
        F.col("destination_airport_id").alias("destination_airport_key"),
        "crs_dep_time",
        "taxi_out",
        "dep_delay",
        "air_time",
        "distance",
        "cancellation_key",
        "temperature",
        "dew_point",
        "rel_humidity",
        "altimeter"
    )
)

Write DataFrame to gold Delta table 

Subsequent runs merge new batch into existing table, only updating records from a newer or equal batch to avoid reprocessing

In [0]:
if not spark.catalog.tableExists(gold_table):

    gold_fact_flights_df_write = (
        gold_fact_flights_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )

else:

    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, gold_table)
    (
        delta_table.alias("t")
        .merge(
            gold_fact_flights_df.alias("s"),
            """t.date_key = s.date_key 
            AND t.mkt_carrier_fl_num = s.mkt_carrier_fl_num 
            AND t.origin_airport_key = s.origin_airport_key 
            AND t.destination_airport_key = s.destination_airport_key
            AND t.tail_number_key = s.tail_number_key
            AND t.departure_timestamp = s.departure_timestamp
            """
        )
        .whenMatchedUpdate(
            set={
                "date_key": "s.date_key",
                "departure_timestamp": "s.departure_timestamp",
                "mkt_unique_carrier_key": "s.mkt_unique_carrier_key",
                "mkt_carrier_fl_num": "s.mkt_carrier_fl_num",
                "op_unique_carrier_key": "s.op_unique_carrier_key",
                "op_carrier_fl_num": "s.op_carrier_fl_num",
                "tail_number_key": "s.tail_number_key",
                "origin_airport_key": "s.origin_airport_key",
                "destination_airport_key": "s.destination_airport_key",
                "crs_dep_time": "s.crs_dep_time",
                "taxi_out": "s.taxi_out",
                "dep_delay": "s.dep_delay",
                "air_time": "s.air_time",
                "distance": "s.distance",
                "cancellation_key": "s.cancellation_key",
                "temperature": "s.temperature",
                "dew_point": "s.dew_point",
                "rel_humidity": "s.rel_humidity",
                "altimeter": "s.altimeter",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    ) 

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-7582106619469047>, line 52
     13 from delta.tables import DeltaTable
     15 delta_table = DeltaTable.forName(spark, gold_table)
     16 (
     17     delta_table.alias("t")
     18     .merge(
     19         gold_fact_flights_df.alias("s"),
     20         """t.date_key = s.date_key 
     21         AND t.mkt_carrier_fl_num = s.mkt_carrier_fl_num 
     22         AND t.origin_airport_key = s.origin_airport_key 
     23         AND t.destination_airport_key = s.destination_airport_key
     24         AND t.tail_number_key = s.tail_number_key
     25         AND t.departure_timestamp = s.departure_timestamp
     26         """
     27     )
     28     .whenMatchedUpdate(
     29         set={
     30             "date_key": "s.date_key",
     31             "departure_timestamp": "s.departure_timestamp",
     32        